# import libraries

In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [2]:
from transformers import AutoTokenizer
from circuit_tracer import ReplacementModel, attribute
from circuit_tracer.utils import create_graph_files
import torch
import torch.nn.functional as F

In [3]:
from huggingface_hub import hf_hub_download

WIDTH = "16k"
L0 = "small"

transcoder_paths = {}
for layer in range(34):
    path = hf_hub_download(
        repo_id="google/gemma-scope-2-4b-pt",
        filename=f"transcoder_all/layer_{layer}_width_{WIDTH}_l0_{L0}/params.safetensors"
    )
    transcoder_paths[layer] = path

In [ ]:
from circuit_tracer.transcoder.single_layer_transcoder import load_transcoder_set
import torch

transcoder_set = load_transcoder_set(
    transcoder_paths=transcoder_paths,
    scan="gemma-scope-2-4b-pt",
    feature_input_hook="hook_resid_mid",
    feature_output_hook="hook_mlp_out",
    device=torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    lazy_encoder=False,
    lazy_decoder=True,
    special_load_fn="gemma-scope-2",
)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-4b-pt")

model = ReplacementModel.from_pretrained_and_transcoders(
    model_name="google/gemma-3-4b-pt",
    transcoders=transcoder_set,
    backend="transformerlens",
    dtype=torch.bfloat16,
    device=torch.device("cuda" if torch.cuda.is_available() else "cpu"),
)

# 1. get top influential features

In [6]:
import json
import glob
from collections import defaultdict

In [7]:
GRAPH_DIR = "./graphs/gemma-3-4b"  # ← set this
DESCRIBED_LAYERS = {9, 17, 22, 29}

In [8]:
def parse_local_feat(node):
    """Extract (layer_int, local_feat_idx) from jsNodeId."""
    js = node.get("jsNodeId", "")
    try:
        layer_feat, _ = js.rsplit("-", 1)
        layer_str, feat_str = layer_feat.split("_", 1)
        return int(layer_str), int(feat_str)
    except Exception:
        return None, None

In [25]:
all_features = defaultdict(float)
 
for fpath in sorted(glob.glob(f"{GRAPH_DIR}/step-*.json")):
    with open(fpath) as f:
        data = json.load(f)
    for node in data.get("nodes", []):
        if "transcoder" not in node.get("feature_type", ""):
            continue
        layer, local_feat = parse_local_feat(node)
        if layer is None:
            continue
        inf = abs(node.get("influence", 0) or 0)
        all_features[(layer, local_feat)] += inf

### top ten from all layers

In [ ]:
top_10_overall = sorted(all_features.items(), key=lambda x: -x[1])[:10]
 
print("=" * 70)
print("TOP 10 OVERALL (all layers)")
print("=" * 70)
for i, ((layer, feat), score) in enumerate(top_10_overall, 1):
    in_desc = "✓ DESCRIBED" if layer in DESCRIBED_LAYERS else "  undescribed"
    print(f"{i:2d}. L{layer:2d} F{feat:5d}  influence={score:8.4f}  {in_desc}")

### top ten from only described layers

In [ ]:
described_features = {lf: s for lf, s in all_features.items() if lf[0] in DESCRIBED_LAYERS}
top_10_described_sorted = sorted(described_features.items(), key=lambda x: -x[1])[:10]
 
print("=" * 70)
print("TOP 10 FROM DESCRIBED LAYERS ONLY (9, 17, 22, 29)")
print("=" * 70)
if top_10_described_sorted:
    for i, ((layer, feat), score) in enumerate(top_10_described_sorted, 1):
        print(f"{i:2d}. L{layer:2d} F{feat:5d}  influence={score:8.4f}")
else:
    print("(none found)")

top_10_described = [lf for lf, _ in top_10_described_sorted]

### top ten from every other layer

In [ ]:
other_features = {lf: s for lf, s in all_features.items() if lf[0] not in DESCRIBED_LAYERS}
top_10_undescribed_sorted = sorted(other_features.items(), key=lambda x: -x[1])[:10]
 
print("=" * 70)
print("TOP 10 FROM OTHER LAYERS (not 9, 17, 22, 29)")
print("=" * 70)
if top_10_undescribed_sorted:
    for i, ((layer, feat), score) in enumerate(top_10_undescribed_sorted, 1):
        print(f"{i:2d}. L{layer:2d} F{feat:5d}  influence={score:8.4f}")
else:
    print("(none found)")

top_10_undescribed = [lf for lf, _ in top_10_undescribed_sorted]

### summary

In [ ]:
print(f"Total unique transcoder features: {len(all_features)}")
print(f"Features in described layers:     {len(described_features)}")
print(f"Features in other layers:         {len(other_features)}")

# intervention

### all 10 features suppressed

In [14]:
# Your prompt
prompt = "A rhyming couplet:\nHe saw a carrot and had to grab it,\n"

# Target position (typically -1 for last token, or specify the rhyme token position)
TARGET_POS = -1

In [ ]:
features_to_intervene = [lf for lf, _ in top_10_overall]
intervention_tuples = [(layer, TARGET_POS, feat, 0.0) for layer, feat in features_to_intervene]

print("\n" + "=" * 70)
print("INTERVENTION TUPLES (top 10 features, suppressed at target position)")
print("=" * 70)
for i, (layer, pos, feat, val) in enumerate(intervention_tuples, 1):
    print(f"{i:2d}. Layer {layer:2d}, Position {pos:3d}, Feature {feat:5d}, Value {val}")

In [ ]:
print("\n" + "=" * 70)
print("GENERATING WITH ORIGINAL FEATURES (no intervention)")
print("=" * 70)

pre_intervention_text = model.feature_intervention_generate(
    prompt,
    [],  # empty list = no interventions
    do_sample=False,
    verbose=True
)[0]

print(f"\nGenerated text:\n{pre_intervention_text}")

# ─────────────────────────────────────────────────────────────
# Generate with all 20 features suppressed
# ─────────────────────────────────────────────────────────────

print("\n" + "=" * 70)
print("GENERATING WITH ALL 10 FEATURES SUPPRESSED")
print("=" * 70)

post_intervention_text = model.feature_intervention_generate(
    prompt,
    intervention_tuples,
    do_sample=False,
    verbose=True
)[0]

print(f"\nGenerated text:\n{post_intervention_text}")

In [ ]:
print("\n" + "=" * 70)
print("COMPARISON: PRE vs POST INTERVENTION")
print("=" * 70)

from circuit_tracer.utils.demo_utils import display_generations_comparison

display_generations_comparison(
    prompt,
    [pre_intervention_text],
    [post_intervention_text]
)

### get pseudo clerp description

In [ ]:
def pseudo_clerp_topk(model, layer, local_feat, tokenizer, top_k=10):
    """
    Project feature decoder direction through unembedding matrix.
    Returns top-k predicted tokens for this feature.
    """
    tc = model.transcoders[layer]
    # W_dec is the direction in residual space the feature 'writes' to
    W_dec = tc.W_dec[local_feat] # shape: [d_model]
    
    with torch.no_grad():
        # model.unembed.W_U shape is usually [d_model, vocab_size]
        W_U = model.unembed.W_U 
        
        # We want to project W_dec (d_model) onto each vocab entry (d_model)
        # So we do: [vocab_size, d_model] @ [d_model]
        # We use W_U.T to get [vocab_size, d_model]
        logits = torch.matmul(W_U.T, W_dec.to(W_U.dtype)) # result: [vocab_size]
    
    # Get top-k tokens
    top_ids = logits.topk(top_k).indices.tolist()
    top_tokens = [tokenizer.decode([i]) for i in top_ids]
    
    return top_tokens

# ─────────────────────────────────────────────────────────────
# Run pseudo-clerp for all top 10 overall features
# ─────────────────────────────────────────────────────────────

print("=" * 70)
print("PSEUDO-CLERP FOR TOP 10 OVERALL FEATURES")
print("=" * 70)

for i, ((layer, feat), score) in enumerate(top_10_overall, 1):
    tokens = pseudo_clerp_topk(model, layer, feat, tokenizer, top_k=10)
    
    # Check if this is a described feature
    in_desc = "✓ DESCRIBED" if layer in DESCRIBED_LAYERS else "  undescribed"
    
    print(f"\n{i:2d}. L{layer:2d} F{feat:5d} (influence={score:8.4f}) {in_desc}")
    print(f"    Top tokens: {tokens}")

In [19]:
# Pseudo-clerp descriptions for top 10 overall features from Gemini 3 (fast)
pseudo_clerps = {
(29, 1784): "Swedish and Northern European capitalized sentence-starting words.",
(32, 156): "Cuneiform characters and ancient Sumerian/Akkadian script tokens.",
(29, 2081): "HTML formatting tags and structural markup elements.",
(29, 107): "Unused internal tokens and bibliography reference markers.",
(31, 1432): "Hebrew alphabet characters and Thai script prefixes.",
(28, 3512): "Mixed technical strings including string variables and MathML.",
(26, 3121): "Mathematical slash symbols and fraction division delimiters.",
(31, 5799): "Concrete singular nouns representing physical objects.",
(33, 294): "Ancient Mesopotamian pictographic and cuneiform logograms.",
(33, 10133): "Common multilingual word fragments and grammatical prefixes.",
}

### one by one suppression

In [ ]:
print("Suppressing one feature at a time to isolate effects...")

ablation_results = []

for layer, feat in features_to_intervene:  # all 10 features
    single_intervention = [(layer, TARGET_POS, feat, 0.0)]
    
    try:
        ablated_text = model.feature_intervention_generate(
            prompt,
            single_intervention,
            do_sample=False,
            verbose=False
        )[0]
        
        # Check if output changed
        changed = ablated_text != pre_intervention_text
        status = "CHANGED" if changed else "unchanged"
        
        # Get clerp (try pseudo_clerps first, fall back to clerps)
        clerp_desc = pseudo_clerps.get((layer, feat)) or clerps.get((layer, feat))
        clerp_str = f" | {clerp_desc}" if clerp_desc else ""
        
        ablation_results.append({
            'layer': layer,
            'feat': feat,
            'changed': changed,
            'text': ablated_text,
            'clerp': clerp_desc
        })
        
        print(f"\nL{layer:2d} F{feat:5d}: {status}{clerp_str}")
        if changed:
            print(f"  → {ablated_text}")
    
    except Exception as e:
        print(f"L{layer:2d} F{feat:5d}: ERROR ({str(e)[:50]})")

# circuit tracing

In [ ]:
# Find "it" in the second line
baseline = "A rhyming couplet:\nHe saw a carrot and had to grab it,\nHe ate it and then he had to crap it"
tokens = model.to_tokens(baseline)
token_strings = [model.tokenizer.decode([t]) for t in tokens[0]]

print("Tokens:")
for i, ts in enumerate(token_strings):
    print(f"  {i}: {repr(ts)}")

# Find the LAST "it" (the one at the end of the second line)
it_positions = [i for i, ts in enumerate(token_strings) if ts.strip() == 'it']
print(f"\n'it' appears at positions: {it_positions}")
rhyme_pos = it_positions[-1]  # last one
print(f"Rhyme token 'it' at position {rhyme_pos}")